### Realizamos la conexion hacia nuestro ADLS por medio de service principal

In [0]:
%run ../config/Access_ADLS_Service_Principal

In [0]:
%run ../includes/configuration

In [0]:
%run ../includes/common_functions

In [0]:
dbutils.widgets.text("p_environment", "")
v_environment = dbutils.widgets.get("p_environment")

### Leemos los archivos csv de nuestro contenedor bronze

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
production_company_schema = StructType(fields=[
    StructField("company_id", IntegerType(), True),
    StructField("company_name", StringType(), True),
])

In [0]:
# Se usa el * para definir que cargue todos los archivos que se llamen production_company
df = spark.read \
    .schema(production_company_schema) \
    .csv(f"{bronze_folder_path}/production_company")

###### Adiccionamos dos nuevas columnas, una para guardar la fecha de ingestion y la otra para guardar el ambiente

In [0]:
df_add = add_columnas_control(df,v_environment)

###### Escribimos los datos en nuestro contenedor silver del data lake

In [0]:
df_add.write.mode("overwrite").parquet(f"{silver_folder_path}/production_company")